In [ ]:
from pynq import Overlay
from client import NPU
import pandas as pd
from IPython.display import display
overlay = Overlay('board.bit')
npu = NPU(overlay)
npu.status()

In [ ]:
weights = [[2, -1, 3, 4], [1, 2, -2, 0], [-3, 1, 1, 2], [0, 4, -1, 3]]
inputs = [5, 0, -2, 1]
result = npu.run(weights, inputs, mode='linear')
display(pd.DataFrame(result['rows']))
print('PASS' if result['passed'] else 'FAIL', 'CPU checks:', result['cpu_checks_passed'], 'comparison checks:', result['comparison_checks_passed'])
assert result['passed']

In [ ]:
summary = []
for mode in ('linear', 'relu', 'tanh', 'sigmoid'):
    result = npu.run(weights, inputs, mode=mode)
    display(pd.DataFrame(result['rows']).assign(mode=mode))
    summary.append({'mode': mode, 'passed': result['passed'], 'cycles_including_SPI': result['job_cycles'], 'CPU_checks': result['cpu_checks_passed'], 'comparison_checks': result['comparison_checks_passed']})
display(pd.DataFrame(summary))
assert all(row['passed'] for row in summary)

In [ ]:
result = npu.run(weights, inputs, mode='tanh', min_iter=6, max_iter=10, full_iter=14, sample_rate=1, threshold=0)
display(pd.DataFrame(result['rows']))
print({key: value for key, value in result.items() if key != 'rows'})
assert result['passed']

In [ ]:
print('All notebook cases passed. Actual results were returned by the SPI-connected NPU through CV32E40P.')
print('Values use the hardware integer representation; activation values have 14 fractional bits.')
print('Job cycles include configuration, SPI transfers and result collection at 25 MHz.')
print('Adaptive runs with unsampled outputs or nonzero thresholds report differences without claiming exact equality.')